**Defining all the datasets**

In [0]:
RAW_VOLUME = "/Volumes/patient_kg_dev/landing/synthea_raw"
BRONZE_SCHEMA = "patient_kg_dev.bronze"

DATASETS = {
    "patients": "patients.csv",
    "encounters": "encounters.csv",
    "conditions": "conditions.csv",
    "medications": "medications.csv",
    "procedures": "procedures.csv",
    "observations": "observations.csv",
    "careplans": "careplans.csv",
}

**Add the ingestion function**

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F


def ingest_bronze_dataset(
    dataset_name: str,
    source_file: str,
    run_id: str,
):
    source_path = f"{RAW_VOLUME}/{source_file}"
    target_table = f"{BRONZE_SCHEMA}.{dataset_name}"

    source_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "FAILFAST")
        .option("nullValue", "__SOURCE_NULL__")
        .csv(source_path)
    )

    source_columns = source_df.columns

    bronze_df = (
        source_df
        .select(
            "*",
            F.col("_metadata.file_name").alias("_source_file"),
            F.col("_metadata.file_path").alias("_source_file_path"),
            F.col("_metadata.file_size").alias("_source_file_size"),
            F.col("_metadata.file_modification_time").alias(
                "_source_file_modified_at"
            ),
        )
        .withColumn(
            "_row_content_sha256",
            F.sha2(
                F.to_json(
                    F.struct(
                        *[F.col(column) for column in source_columns]
                    ),
                    options={"ignoreNullFields": "false"},
                ),
                256,
            ),
        )
        .withColumn("_ingestion_run_id", F.lit(run_id))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    source_count = source_df.count()

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    target_count = spark.table(target_table).count()

    if source_count != target_count:
        raise RuntimeError(
            f"Count mismatch for {dataset_name}: "
            f"source={source_count}, target={target_count}"
        )

    return {
        "dataset": dataset_name,
        "source_file": source_file,
        "target_table": target_table,
        "source_count": source_count,
        "target_count": target_count,
        "status": "PASS",
    }

**Run all seven datasets**

In [0]:
run_id = str(uuid4())
results = []

for dataset_name, source_file in DATASETS.items():
    print(f"Ingesting {dataset_name}...")

    result = ingest_bronze_dataset(
        dataset_name=dataset_name,
        source_file=source_file,
        run_id=run_id,
    )

    results.append(result)

print(f"Completed Bronze ingestion run: {run_id}")

**Display the results**

In [0]:
results_df = spark.createDataFrame(results)

display(results_df.orderBy("dataset"))

**Verify all tables exist**

In [0]:
%sql
SHOW TABLES IN patient_kg_dev.bronze;

In [0]:
%sql
SELECT
    _source_file,
    COUNT(*) AS row_count,
    COUNT(_row_content_sha256) AS populated_hashes,
    COUNT(DISTINCT _ingestion_run_id) AS ingestion_runs
FROM patient_kg_dev.bronze.observations
GROUP BY _source_file;

**Check Duplicate Hashes:**

In [0]:
%sql
SELECT
    _row_content_sha256,
    COUNT(*) AS occurrence_count
FROM patient_kg_dev.bronze.observations
GROUP BY _row_content_sha256
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

In [0]:
%sql
SELECT COUNT(*)
FROM patient_kg_dev.bronze.observations;

VALIDATION

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(_source_file) AS source_file_populated,
    COUNT(_row_content_sha256) AS hash_populated
FROM patient_kg_dev.bronze.observations;

In [0]:
%sql
SELECT 'patients' AS dataset, COUNT(*) AS rows
FROM patient_kg_dev.bronze.patients

UNION ALL
SELECT 'encounters', COUNT(*)
FROM patient_kg_dev.bronze.encounters

UNION ALL
SELECT 'conditions', COUNT(*)
FROM patient_kg_dev.bronze.conditions

UNION ALL
SELECT 'medications', COUNT(*)
FROM patient_kg_dev.bronze.medications

UNION ALL
SELECT 'procedures', COUNT(*)
FROM patient_kg_dev.bronze.procedures

UNION ALL
SELECT 'observations', COUNT(*)
FROM patient_kg_dev.bronze.observations

UNION ALL
SELECT 'careplans', COUNT(*)
FROM patient_kg_dev.bronze.careplans;

Cohort rule confirmation

In [0]:
%sql
SELECT
    SYSTEM,
    CODE,
    DESCRIPTION,
    COUNT(*) AS condition_rows,
    COUNT(DISTINCT PATIENT) AS patient_count
FROM patient_kg_dev.bronze.conditions
WHERE SYSTEM = 'http://snomed.info/sct'
  AND CODE = '414545008'
GROUP BY SYSTEM, CODE, DESCRIPTION;

In [0]:
%sql
SELECT DISTINCT PATIENT
FROM patient_kg_dev.bronze.conditions
WHERE SYSTEM = 'http://snomed.info/sct'
  AND CODE = '414545008'
ORDER BY PATIENT;